In [ ]:
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn

!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PHASE1_CHECKPOINT = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"

Mounted at /content/drive


In [ ]:
import pandas as pd, glob, os, re

AUG_BASE_DIR = "/content/drive/MyDrive/final_project/Shemo_Augmented"
csv_path = glob.glob(f"{AUG_BASE_DIR}/*.csv")[0]
df = pd.read_csv(csv_path)
df["base_id"] = df["path"].apply(lambda p: os.path.splitext(os.path.basename(p))[0])

EMOTION_MAP = {"ANGRY": "anger", "HAPPY": "happiness", "HAPPINESS": "happiness",
               "NEUTRAL": "neutral", "SAD": "sadness", "SADNESS": "sadness"}
label_lookup = df.drop_duplicates("base_id").set_index("base_id")["emotion"].to_dict()

AUDIO_AUG_DIR = f"{AUG_BASE_DIR}/shemo_augmented"
aug_files = glob.glob(f"{AUDIO_AUG_DIR}/**/*.wav", recursive=True)

def get_base_id_from_audio_filename(fp):
    return re.sub(r'_aug\d+$', '', os.path.splitext(os.path.basename(fp))[0])

labeled_files = []
for f in aug_files:
    base_id = get_base_id_from_audio_filename(f)
    raw_emo = label_lookup.get(base_id)
    if raw_emo is None:
        continue
    emo = EMOTION_MAP.get(raw_emo.upper())
    if emo:
        labeled_files.append((f, emo))

print(f"Number of augmented samples for replay: {len(labeled_files)}")

Number of augmented samples for replay: 6727


In [ ]:
REAL_DATA_DIR = "/content/drive/MyDrive/final_project/voice_dataset/audio"
CONVERTED_DIR = "/content/real_audio_wav"

# Install ffmpeg for audio conversion
!apt-get -y -q install ffmpeg > /dev/null 2>&1
import subprocess

# Convert all .m4a files to .wav (16kHz, mono)
m4a_files = glob.glob(f"{REAL_DATA_DIR}/**/*.m4a", recursive=True)
os.makedirs(CONVERTED_DIR, exist_ok=True)
for f in m4a_files:
    speaker = os.path.basename(os.path.dirname(f))
    out_dir = os.path.join(CONVERTED_DIR, speaker)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".wav")
    if not os.path.exists(out_path):
        subprocess.run(["ffmpeg", "-y", "-i", f, "-ar", "16000", "-ac", "1", out_path],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Map emotion codes to standardized labels
LABEL_CODE_MAP = {"ANG": "anger", "HAP": "happiness", "NEU": "neutral", "SAD": "sadness"}
def get_label_from_filename(fp):
    name = os.path.splitext(os.path.basename(fp))[0].upper()
    for code, emo in LABEL_CODE_MAP.items():
        if code in name:
            return emo
    return None

# Load real data with labels and speaker IDs
speaker_dirs = sorted(glob.glob(f"{CONVERTED_DIR}/speaker_*"), key=lambda p: int(p.split("_")[-1]))
real_data = []  # (path, label, speaker_id)
for d in speaker_dirs:
    sid = os.path.basename(d)
    for f in glob.glob(f"{d}/*.wav"):
        lb = get_label_from_filename(f)
        if lb:
            real_data.append((f, lb, sid))

print(f"Number of real samples: {len(real_data)}")

Number of real samples: 112


In [ ]:
from transformers import AutoConfig, Wav2Vec2FeatureExtractor
import torchaudio
from torch.utils.data import Dataset
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from functools import partial

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(PHASE1_CHECKPOINT)
target_sampling_rate = feature_extractor.sampling_rate

NUM_LAYERS_TO_UNFREEZE = 2
N_EPOCHS = 4
LR = 2e-5
BATCH_SIZE = 4
REPLAY_SAMPLES_PER_CLASS = 60

EMOTIONS = ["anger", "happiness", "neutral", "sadness"]

def speech_file_to_array_fn(path, target_sr):
    speech_array, orig_sr = torchaudio.load(path)
    if speech_array.shape[0] > 1:
        speech_array = speech_array.mean(dim=0, keepdim=True)
    return torchaudio.transforms.Resample(orig_sr, target_sr)(speech_array).squeeze().numpy()

class SERDataset(Dataset):
    def __init__(self, pairs, target_sr):
        self.pairs = pairs
        self.target_sr = target_sr
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        speech = speech_file_to_array_fn(path, self.target_sr)
        return {"speech": speech, "label": label2id[label]}

def collate_fn(batch, feature_extractor, sr):
    speeches = [b["speech"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch])
    inputs = feature_extractor(speeches, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs["labels"] = labels
    return inputs

label2id = {l: i for i, l in enumerate(EMOTIONS)}
id2label = {i: l for i, l in enumerate(EMOTIONS)}

def build_fresh_model():
    config = AutoConfig.from_pretrained(PHASE1_CHECKPOINT)
    m = Wav2Vec2ForSpeechClassification.from_pretrained(PHASE1_CHECKPOINT, config=config).to(device)
    m.wav2vec2.feature_extractor._freeze_parameters()
    total_layers = len(m.wav2vec2.encoder.layers)
    for i, layer in enumerate(m.wav2vec2.encoder.layers):
        grad = i >= (total_layers - NUM_LAYERS_TO_UNFREEZE)
        for p in layer.parameters():
            p.requires_grad = grad
    return m

def train_one_fold(train_pairs):
    model = build_fresh_model()
    model.train()
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    scaler = GradScaler(enabled=torch.cuda.is_available())

    dataset = SERDataset(train_pairs, target_sampling_rate)
    loader = torch.utils.data.DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=partial(collate_fn, feature_extractor=feature_extractor, sr=target_sampling_rate)
    )

    for epoch in range(N_EPOCHS):
        total_loss = 0
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        print(f"    epoch {epoch+1}/{N_EPOCHS} - loss: {total_loss/len(loader):.4f}")

    model.eval()
    return model

@torch.no_grad()
def predict_with_model(m, path):
    speech = speech_file_to_array_fn(path, target_sampling_rate)
    inputs = feature_extractor(speech, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    logits = m(input_values).logits
    pred_id = torch.argmax(logits, dim=-1).item()
    return id2label[pred_id]

In [ ]:
import random
random.seed(42)

QUICK_TEST = False

# Select which speakers to run (first 2 if quick test, otherwise all)
unique_speakers = sorted(set(s for _, _, s in real_data), key=lambda x: int(x.split("_")[-1]))
speakers_to_run = unique_speakers[:2] if QUICK_TEST else unique_speakers

# Prepare replay samples pool by class
replay_pool_by_class = {emo: [f for f, l in labeled_files if l == emo] for emo in EMOTIONS}

all_true_deep, all_pred_deep = [], []

# Leave-one-speaker-out cross-validation with replay augmentation
for fold_i, test_speaker in enumerate(speakers_to_run):
    print(f"\n=== Fold {fold_i+1}/{len(speakers_to_run)} — Testing on {test_speaker} ===")

    # Split real data by speaker
    train_real = [(f, l) for f, l, s in real_data if s != test_speaker]
    test_real = [(f, l) for f, l, s in real_data if s == test_speaker]

    # Sample replay files from each class
    replay_samples = []
    for emo, files in replay_pool_by_class.items():
        chosen = random.sample(files, min(REPLAY_SAMPLES_PER_CLASS, len(files)))
        replay_samples.extend([(f, emo) for f in chosen])

    # Combine real training data with replay samples
    train_pairs = train_real + replay_samples
    print(f"    train: {len(train_pairs)} (real: {len(train_real)} + replay: {len(replay_samples)}) | test: {len(test_real)}")

    # Train model for this fold
    fold_model = train_one_fold(train_pairs)

    # Evaluate on test speaker
    for path, true_label in test_real:
        pred = predict_with_model(fold_model, path)
        all_true_deep.append(true_label)
        all_pred_deep.append(pred)
        print(f"    {os.path.basename(path)}: true={true_label} | prediction={pred}")

    # Clean up
    del fold_model
    torch.cuda.empty_cache()

print("\nAll folds completed.")


=== Fold 1/26 — Testing on speaker_1 ===
    train: 347 (real: 107 + replay: 240) | test: 5


/tmp/ipykernel_825/2307248473.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_825/2307248473.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


    epoch 1/4 - loss: 0.8075
    epoch 2/4 - loss: 0.6687
    epoch 3/4 - loss: 0.6582
    epoch 4/4 - loss: 0.6253
    025_ANG.wav: true=anger | prediction=happiness
    077_HAP.wav: true=happiness | prediction=neutral
    023_HAP.wav: true=happiness | prediction=neutral
    010_SAD.wav: true=sadness | prediction=sadness
    024_NEU.wav: true=neutral | prediction=neutral

=== Fold 2/26 — Testing on speaker_2 ===
    train: 346 (real: 106 + replay: 240) | test: 6
    epoch 1/4 - loss: 0.7509
    epoch 2/4 - loss: 0.6435
    epoch 3/4 - loss: 0.6202
    epoch 4/4 - loss: 0.5942
    076_ANG.wav: true=anger | prediction=happiness
    050_HAP.wav: true=happiness | prediction=happiness
    052_NEU.wav: true=neutral | prediction=neutral
    003_ANG.wav: true=anger | prediction=anger
    075_SAD.wav: true=sadness | prediction=sadness
    051_SAD.wav: true=sadness | prediction=happiness

=== Fold 3/26 — Testing on speaker_3 ===
    train: 350 (real: 110 + replay: 240) | test: 2
    epoch 1/4 -

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

print(f"Accuracy (deep fine-tune): {accuracy_score(all_true_deep, all_pred_deep):.4f}")
print(classification_report(all_true_deep, all_pred_deep))

# Display confusion matrix
labels_order = sorted(set(all_true_deep) | set(all_pred_deep))
cm = confusion_matrix(all_true_deep, all_pred_deep, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

Accuracy (deep fine-tune): 0.4732
              precision    recall  f1-score   support

       anger       0.71      0.19      0.29        27
   happiness       0.52      0.43      0.47        30
     neutral       0.39      0.67      0.49        27
     sadness       0.50      0.61      0.55        28

    accuracy                           0.47       112
   macro avg       0.53      0.47      0.45       112
weighted avg       0.53      0.47      0.45       112



,anger,happiness,neutral,sadness
anger,5,6,11,5
happiness,2,13,11,4
neutral,0,1,18,8
sadness,0,5,6,17
